# 课程 02 - 探索 Microsoft Agent 框架

**Microsoft Agent 框架（MAF）** 是一个用于构建 AI 代理的统一框架。它提供了一个简洁、可组合的架构，包含四个核心构建模块：

- <strong>客户端</strong> – 连接到 AI 模型端点并处理通信
- <strong>代理</strong> – 包装客户端，带有指令和工具定义
- <strong>工具</strong> – 通过模型可调用的自定义函数扩展代理能力
- <strong>会话</strong> – 维护多轮交互的对话历史

在本课中，我们将构建一个使用这些概念来检查目的地可用性的<strong>旅行预订代理</strong>。


## 设置


In [ ]:
# Install the Microsoft Agent Framework package
! pip install agent-framework azure-ai-projects -U -q
! pip install python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import asyncio
import dotenv
from typing import Annotated

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential

dotenv.load_dotenv(dotenv.find_dotenv())

## 理解 Agent 框架架构

Microsoft Agent 框架遵循分层架构：

```
Client  →  Agent  →  Tools
                  →  Session
```

1. <strong>客户端</strong> – `FoundryChatClient` 连接到 Azure OpenAI 部署，处理身份验证、请求格式化和响应解析。
2. **Agent** – 通过 `provider.create_agent()` 从客户端创建，agent 结合了模型访问、指令（系统提示）和工具。
3. <strong>工具</strong> – 用 `@tool` 装饰的 Python 函数，agent 可以调用它们执行操作或获取数据。
4. <strong>会话</strong> – `AgentSession` 对象（通过 `agent.create_session()` 创建），存储对话历史，实现多轮对话，agent 记忆之前的上下文。

让我们一步步构建每一层。


In [ ]:
# Create the client – this is the connection to the AI model
endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
model = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

if not endpoint or not model:
    raise ValueError(
        "Missing required environment variables. "
        "Please set AZURE_AI_PROJECT_ENDPOINT and AZURE_AI_MODEL_DEPLOYMENT_NAME as environment variables (e.g., in your .env file or shell environment)."
    )

provider = FoundryChatClient(
    project_endpoint=endpoint,
    model=model,
    credential=AzureCliCredential()
)

## 使用 @tool 装饰器添加工具

工具让代理可以执行生成文本以外的操作。`@tool` 装饰器将普通的 Python 函数转换为代理可以调用的功能。

关键点：
- 使用 `Annotated[type, "description"]`，让模型理解每个参数。
- 文档字符串变成模型看到的工具描述。
- `approval_mode="never_require"` 表示工具自动运行，无需用户确认。


In [ ]:
@tool(approval_mode="never_require")
def check_destination_availability(
    destination: Annotated[str, "The destination to check availability for"]
) -> str:
    """Check if a vacation destination is currently available for booking."""
    available = {
        "Barcelona": True,
        "Tokyo": True,
        "Cape Town": False,
        "Vancouver": True,
        "Dubai": False,
    }
    is_available = available.get(destination, False)
    return f"{destination} is {'available' if is_available else 'not available'} for booking."

## 使用工具创建代理

现在我们将客户端、指令和工具组合成一个代理。`instructions` 作为系统提示——它们定义了代理的角色和行为。


In [ ]:
agent = provider.as_agent(
    name="TravelAvailabilityAgent",
    instructions=(
        "You are a travel booking agent. Help users check destination availability "
        "and make recommendations. Always check availability before recommending a destination."
    ),
    tools=[check_destination_availability],
)

## 多轮对话会话

`AgentSession`（通过 `agent.create_session()` 创建）跟踪对话中的所有消息。通过在每次 `agent.run()` 调用中传递相同的会话，代理可以访问完整的对话历史并引用之前的消息。

我们传入 `tools=[check_destination_availability]`，以便代理在每轮中都能调用我们的可用性检查器。


In [ ]:
session = agent.create_session()

# Turn 1: Ask about available destinations
response = await agent.run(
    "Which destinations do you have available?",
    session=session,
)
print(f"Agent: {response}")

In [ ]:
# Turn 2: Follow-up question — the agent remembers the conversation
response = await agent.run(
    "I'd like to go somewhere warm. What's available?",
    session=session,
)
print(f"Agent: {response}")

## 总结

在本课中，您探索了微软代理框架的四大支柱：

| 概念 | 你学到了什么 |
|---------|------------------|
| <strong>客户端</strong> | `FoundryChatClient` 使用基于凭证的认证连接到 Azure OpenAI |
| <strong>代理</strong> | `provider.create_agent()` 将模型连接与指令和名称绑定在一起 |
| <strong>工具</strong> | `@tool` 装饰器暴露 Python 函数供代理调用 |
| <strong>会话</strong> | `agent.create_session()` 跨多轮保持对话历史 |

这些构建模块组合在一起，创建能够进行自然对话、调用外部函数并保持上下文的代理 —— 这是后续课程中更高级代理模式的基础。


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免责声明**：
本文件由 AI 翻译服务 [Co-op Translator](https://github.com/Azure/co-op-translator) 翻译完成。尽管我们力求准确，但请注意，自动翻译可能包含错误或不准确之处。原始语言版文件应视为权威来源。对于重要信息，建议使用专业人工翻译。我们对因使用本翻译而产生的任何误解或误释不承担责任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
